# Advanced PySpark Optimization — Examples 11–20

Notebook 2 of the 5-notebook Advanced PySpark Optimization series.

## Examples

11. Partitioned writes with `partitionBy`
12. Partition pruning
13. Bucketing with `bucketBy`
14. Bucketing vs. repartitioning
15. Small files vs. partition/file sizing
16. Where shuffles come from
17. `groupBy()` performance behavior
18. `groupByKey()` vs. `reduceByKey()`
19. Multi-level aggregation
20. `rollup()` and `cube()`

The CSV inputs are intentionally tiny. This notebook demonstrates **mechanics and execution plans**, not production-scale benchmark numbers.

## Source mapping

This notebook combines the source material's discussions of physical partitioning, partition pruning, bucketing, output-file sizing, shuffle, aggregation behavior, transformation selection, and hierarchical/multidimensional aggregation.

In [ ]:
from pathlib import Path
import shutil

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

BASE_PATH = Path.cwd()
DATA_PATH = BASE_PATH / "data"
if not DATA_PATH.exists():
    candidate = Path("/mnt/data/pyspark_examples_11_20/data")
    if candidate.exists():
        DATA_PATH = candidate

assert DATA_PATH.exists(), "Could not find data/. Update BASE_PATH."

WORK_PATH = BASE_PATH / "work"
if str(BASE_PATH).startswith("/mnt/data/pyspark_examples_11_20"):
    WORK_PATH = BASE_PATH / "work"

if WORK_PATH.exists():
    shutil.rmtree(WORK_PATH)
WORK_PATH.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("Advanced-PySpark-Examples-11-20")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.warehouse.dir", str(WORK_PATH / "warehouse"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_date", StringType(), True),
    StructField("order_year", StringType(), True),
    StructField("order_month", StringType(), True),
])

orders_df = (
    spark.read.option("header", True)
    .schema(orders_schema)
    .csv(str(DATA_PATH / "orders.csv"))
)

orders_df.show()
print("Input partitions:", orders_df.rdd.getNumPartitions())
print("Work path:", WORK_PATH)

# Example 11 — Partitioned Writes with `partitionBy`

**Concepts:** physical partitioning, partition-aware writes, choosing partition columns.

We write the dataset as Parquet partitioned by `order_year` and `order_month`.

In [ ]:
partitioned_output = WORK_PATH / "orders_partitioned"

(
    orders_df.write.mode("overwrite")
    .partitionBy("order_year", "order_month")
    .parquet(str(partitioned_output))
)

print("Partitioned output structure:")
for path in sorted(partitioned_output.rglob("*")):
    relative = path.relative_to(partitioned_output)
    if path.is_dir() or path.suffix == ".parquet":
        print(relative)

partitioned_read_df = spark.read.parquet(str(partitioned_output))
partitioned_read_df.printSchema()
partitioned_read_df.orderBy("order_id").show()

# Example 12 — Partition Pruning

**Concepts:** partition filters, partition pruning, reduced scanning.

Filter on the columns used in the physical partition layout and inspect the plan for `PartitionFilters`.

In [ ]:
pruned_df = (
    spark.read.parquet(str(partitioned_output))
    .filter(
        (F.col("order_year") == "2026") &
        (F.col("order_month") == "02")
    )
)

print("PLAN — LOOK FOR PARTITION FILTERS")
pruned_df.explain("formatted")
print("\nFiltered result:")
pruned_df.show()

print("\nCompare with a non-partition-column filter:")
non_partition_filter_df = (
    spark.read.parquet(str(partitioned_output))
    .filter(F.col("category") == "Electronics")
)
non_partition_filter_df.explain("formatted")
non_partition_filter_df.show()

# Example 13 — Bucketing with `bucketBy`

**Concepts:** hash-based bucketing, fixed bucket counts, bucketed tables.

`bucketBy()` is used with table-writing APIs. We create a small table bucketed by `customer_id`.

Optimizer behavior can vary by Spark deployment; this lab focuses on creating and inspecting the layout.

In [ ]:
spark.sql("DROP TABLE IF EXISTS orders_bucketed")

(
    orders_df.write.format("parquet").mode("overwrite")
    .bucketBy(4, "customer_id")
    .sortBy("customer_id")
    .saveAsTable("orders_bucketed")
)

bucketed_df = spark.table("orders_bucketed")
bucketed_df.show()

print("\nTable metadata:")
spark.sql("DESCRIBE EXTENDED orders_bucketed").show(truncate=False)

print("\nPhysical plan:")
bucketed_df.explain("formatted")

# Example 14 — Bucketing vs. Repartitioning

**Concepts:** hash distribution, runtime repartitioning, persisted bucket layout.

`repartition()` redistributes data for the current lineage.

Bucketing is a persisted table layout with a fixed bucket count.

In [ ]:
repartitioned_by_customer = orders_df.repartition(4, "customer_id")

print("Repartitioned DataFrame partitions:", repartitioned_by_customer.rdd.getNumPartitions())

print("\nREPARTITION PLAN")
repartitioned_by_customer.explain("formatted")

print("\nBUCKETED TABLE PLAN")
bucketed_df.explain("formatted")

print("\nCustomer distribution:")
orders_df.groupBy("customer_id").count().orderBy("customer_id").show()

print("Key takeaway:")
print("repartition() = runtime redistribution for this lineage")
print("bucketBy() = persisted table layout with a fixed bucket count")

# Example 15 — Small Files vs. Partition/File Sizing

**Concepts:** partition count, output file count, small-file behavior.

We write the same tiny dataset using different partition counts and count the resulting Parquet files.

In [ ]:
def parquet_files(path):
    return sorted(p for p in Path(path).rglob("*.parquet") if p.is_file())

few_files_path = WORK_PATH / "few_files"
many_files_path = WORK_PATH / "many_files"

orders_df.coalesce(1).write.mode("overwrite").parquet(str(few_files_path))
orders_df.repartition(4).write.mode("overwrite").parquet(str(many_files_path))

few_files = parquet_files(few_files_path)
many_files = parquet_files(many_files_path)

print("coalesce(1) output files:", len(few_files))
for p in few_files:
    print(" ", p.name)

print("\nrepartition(4) output files:", len(many_files))
for p in many_files:
    print(" ", p.name)

print("\nPartition count influences the number of output tasks/files.")

# Example 16 — Where Shuffles Come From

**Concepts:** shuffle-producing operations, joins, aggregations, repartitioning, `Exchange`.

Inspect these physical plans and look for `Exchange`.

In [ ]:
print("1) REPARTITION")
orders_df.repartition(4, "customer_id").explain("formatted")

print("\n2) GROUP BY")
orders_df.groupBy("category").agg(F.sum("amount").alias("total_amount")).explain("formatted")

print("\n3) JOIN")
small_customers = (
    orders_df.select("customer_id").distinct()
    .withColumn("customer_group", F.substring("customer_id", 1, 2))
)
orders_df.join(small_customers, "customer_id").explain("formatted")

# Example 17 — `groupBy()` Performance Behavior

**Concepts:** aggregation-induced shuffle, low/high cardinality grouping, aggregation planning.

On real data, higher-cardinality grouping can create more groups and potentially more memory pressure.

In [ ]:
low_cardinality_agg = (
    orders_df.groupBy("category")
    .agg(F.sum("amount").alias("total_amount"), F.count("*").alias("orders"))
)

high_cardinality_agg = (
    orders_df.groupBy("order_id")
    .agg(F.sum("amount").alias("total_amount"), F.count("*").alias("orders"))
)

print("LOW-CARDINALITY GROUPING PLAN")
low_cardinality_agg.explain("formatted")
low_cardinality_agg.show()

print("\nHIGH-CARDINALITY GROUPING PLAN")
high_cardinality_agg.explain("formatted")
high_cardinality_agg.show()

# Example 18 — `groupByKey()` vs. `reduceByKey()`

**Concepts:** transformation selection and aggregation efficiency.

Both calculate total amount by category. `reduceByKey()` can combine values before shuffle, whereas `groupByKey()` groups all values for a key.

In [ ]:
category_amount_rdd = (
    orders_df.select("category", "amount").rdd
    .map(lambda row: (row["category"], row["amount"]))
)

group_by_key_result = (
    category_amount_rdd.groupByKey()
    .mapValues(lambda values: sum(values))
)

reduce_by_key_result = (
    category_amount_rdd
    .reduceByKey(lambda left, right: left + right)
)

group_by_key_dict = dict(group_by_key_result.collect())
reduce_by_key_dict = dict(reduce_by_key_result.collect())

print("groupByKey result:", sorted(group_by_key_dict.items()))
print("reduceByKey result:", sorted(reduce_by_key_dict.items()))

assert group_by_key_dict == reduce_by_key_dict
print("Validation passed.")

# Example 19 — Multi-Level Aggregation

**Concepts:** grouped aggregation, intermediate aggregation, hierarchical processing.

First aggregate by `(order_month, category)`, then aggregate the smaller result by `category`.

In [ ]:
monthly_category_agg = (
    orders_df.groupBy("order_month", "category")
    .agg(
        F.sum("amount").alias("monthly_category_amount"),
        F.count("*").alias("monthly_orders")
    )
)

category_summary = (
    monthly_category_agg.groupBy("category")
    .agg(
        F.sum("monthly_category_amount").alias("total_amount"),
        F.sum("monthly_orders").alias("total_orders")
    )
)

print("LEVEL 1 — MONTH + CATEGORY")
monthly_category_agg.orderBy("order_month", "category").show()

print("\nLEVEL 2 — CATEGORY")
category_summary.orderBy("category").show()

print("\nPLAN")
category_summary.explain("formatted")

# Example 20 — `rollup()` and `cube()`

**Concepts:** hierarchical aggregation, subtotals, grand totals, multidimensional aggregation.

`rollup()` produces hierarchical totals. `cube()` produces combinations across dimensions.

`NULL` grouping values represent subtotal/grand-total rows generated by the aggregation.

In [ ]:
rollup_df = (
    orders_df.rollup("order_month", "category")
    .agg(F.sum("amount").alias("total_amount"))
    .orderBy("order_month", "category")
)

cube_df = (
    orders_df.cube("order_month", "category")
    .agg(F.sum("amount").alias("total_amount"))
    .orderBy("order_month", "category")
)

print("ROLLUP RESULT")
rollup_df.show()

print("\nCUBE RESULT")
cube_df.show()

print("\nROLLUP PLAN")
rollup_df.explain("formatted")

print("\nCUBE PLAN")
cube_df.explain("formatted")

# Notebook 2 Summary

Examples 11–20 covered the relationship between **data layout and execution cost**:

- `partitionBy`
- partition pruning
- bucketing
- bucketing vs. repartitioning
- output file counts and small-file behavior
- shuffle-producing operations
- aggregation behavior
- `groupByKey()` vs. `reduceByKey()`
- multi-level aggregation
- `rollup`
- `cube`

## Optimization habit

1. Inspect the physical plan.
2. Look for `Exchange` and redistribution.
3. Check partition counts and key distribution.
4. Consider whether data layout can reduce repeated work.
5. Validate correctness after optimization.

## Next notebook

Examples 21–30 will focus on:

- cache vs. no cache
- `cache()` vs. `persist()`
- `MEMORY_AND_DISK`
- storage/execution memory trade-offs
- unnecessary caching
- shuffle spill investigation
- memory configuration concepts
- baseline joins
- broadcast joins
- broadcast thresholds